## Cryptocurrency Options Trading 

In this project we contsruct algorithms to trade BTC/USD options.

- The increasing amount of volume and open interest in the last decade singify that the market for cryptocurrency is becoming more liquid.
- To remain liquid and have reasonable amount of quotes, we only consider ATM options that expire within a month to trade.
- We mainly utilise the Black-Scholes model (assuming Stock pricing follows a geometric brownian motion (BGM)) to build an algorithm that identifies trades which disagrees with the current market, creating an edge. We make use of the historical and implied volatility.
- Moreover, we use a simple dynamic delta hedging, where we see that our strategy proves to be profitable, even during hard periods e.g. 2020 (Covid year). We dynamicaly hedge our trades daily --- the rate can be higher.


In [3]:
import numpy as np
import pandas as pd
import math
from scipy.stats import norm
from scipy.optimize import newton, brentq
import ccxt
import sys
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.dates import DayLocator, DateFormatter
from tabulate import tabulate


binance = ccxt.binance()

First we create a function to fetch data from the *CryptoCurrency eXchange Trading Library cctx*. This function will be used throughout, e.g. to get closing prices so at to create the historical and implied volatility or to convert bid-ask prices (which are in BTC) to USD.

We also create a simple function to calculate the historical volatility. Recall, that we assume that our Stock prices follow the GBM:
$$S_t = \mu S_tdt + \sigma S_tdW^{\mathbb{P}}.$$

That means that between a time period $\Delta t$ (e.g. day 1 to day 2) our ratio of Stock prices follow lognormal distribution with variance $\sigma^2 \Delta t$, i.e.
$$\ln (S_t/S_{t-1}) \sim N((\mu -\sigma^2/2)\Delta t,\sigma^2 \Delta t)$$

So, if we want to find our annualized historical volatility we just need to find the annual log standard deviation of our ClosePrice data fetched by our previous function:
$$\sigma_{AnualHist} \approx \ln (S_t/S_{t-1})\times \sqrt{T},$$

where $T$ is the number of periods per year (in the case of BTC options $T=365$ for daily frequency | $T=365\times24$ for hourly data).

In [4]:
def GetCryptoData(symbol: str, timeframe: str, candles: int, start: str) -> pd.DataFrame:
    """
    Receive OHLCV data from cctx.
    
    symbol = "BTC/USDT"
    timeframe = "1d"
    limit = 365
    start_date = "2021-01-01"
    """
    #convert start_date to timestamp -- (see https://docs.python.org/3/library/datetime.html#datetime-objects) || *1000 to convert to ms
    since =  int(datetime.timestamp(datetime.strptime(start, "%Y-%m-%d")) * 1000)
    
    #fetch data
    bars = binance.fetch_ohlcv(symbol, timeframe = timeframe, since = since, limit = candles)
    df = pd.DataFrame(bars, columns=["Timestamp", "Open", "High", "Low", "Close", "Volume"])
    df["Timestamp"] = pd.to_datetime(df["Timestamp"], unit= "ms")

    return df

def HistoricalVolCalc(ClosePrice: pd.Series, timestamp: pd.Series, period: str) -> float:
    """
    Calculate annualized historical volatility from OHLCV data.

    Parameters:
    - ClosePrice: Close market prices
    - timestamp: pandas Series of timestamps
    - period: '1d' (for daily data), '1h' (for hourly data), '5m' (for 5m interval data)
    """
    
    log_Change_returns = np.log(ClosePrice / ClosePrice.shift(1)).dropna()
    logStd =  log_Change_returns.std()

    #Calculating Annualised Log Std

    if period == '1d':
        time_diff = 1.
    elif period == '1h':
        time_diff = 1/24
    elif period == '5m':
        time_diff = 1/24 * 5/60
    else:
        raise ValueError(f"Unsupported period: {period}. Use '1d' or '1h' or '5m'.")
    
    annualLogStd = (365/time_diff)**0.5 * logStd
    
    return annualLogStd

In [5]:
def TimeMinPeriods(timestamp: str, interval: int) -> float:
    """
    Parameters:
    - timestamp: The timestamp i.e. the date e.g. "2020-10-01"
    - interval: The fixed-length time intervals

    Returns:
            How many fixed-length time intervals have passed since midnight (beginning of the day)
    """
    #for no.1 of 5 minute periods    
    timestamp_format = '%Y-%m-%d %H:%M:%S.%f'

    # Parse the timestamp string
    timestamp = datetime.strptime(timestamp, timestamp_format)

    # Calculate total minutes from midnight
    total_minutes = timestamp.hour * 60 + timestamp.minute

    # Calculate the number of 5-minute periods
    periods_of_interval_minutes = total_minutes // interval

    return periods_of_interval_minutes

## Black-Scholes formula.

We now code the Black-Scholes formula for European calls and puts--with no dividends. We have the following **notations**: 

$S:$ the price of the underlying stock (Spot price).

$K:$ the strike price.

$C(S,t):$ price of a European call options (at time t).

$P(S,t):$ price of a European put options (at time t).

$t:$ time to expiration.

$r:$ risk-free interest rate.

$\sigma:$ the volatility.

The Black-Scholes formulae are given by

$$ \underline{\text{Call options: }} \ \ C(S,t) = SN(d_1) - K\exp^{-rt}N(d_2),$$
$$ \underline{\text{Put options: }} \ \ P(S,t) = K\exp^{-rt}N(-d_2) - SN(-d_1),$$

where $d_1 = \frac{ln(S/K) + (r+\sigma^2/2)t}{\sigma\sqrt{t}},$ and $d_2 = d_1 -\sigma\sqrt{t}$.

There is also an underlying relationship between the price of a **put** and a **call** option, namely the *Put-call parity*:

$$C + K\exp^{-rt} = P + S.$$

So given price for a **call** option, we can easily output an equivalent price for a **put** option, by rearranging the above equation (and vice-versa).

In [6]:
def BlackScholes(spot_price: float, strike_price: float, T: float, volatility: float, rfr: float, CallorPut: str)-> float:
    """ 
    Calculate the theoretical price of a call and put option using the Black-Scholes model.

    Parameters:
        CallorPut:  Option type. Must be "call" or "put".
        spot_price : Current price of the underlying asset.
        strike_price : Strike price of the option.
        T : Time to maturity --(in years)--.
        volatility : Volatility of the underlying asset (annualized standard deviation).
        rfr : Risk-free interest rate (annualized).
        
    Returns:
        float: The theoretical option price.

    Notes
    -----
    Assumes no dividends and constant volatility and interest rates.
    """    
    
    CallorPut = CallorPut.lower()
    d1 = (np.log(spot_price / strike_price) + (rfr + 0.5 * volatility**2) * T) / (volatility * np.sqrt(T))
    d2 = d1 - volatility * np.sqrt(T)
    if CallorPut == "call":
        option_value = spot_price * norm.cdf(d1) - strike_price * np.exp(-rfr * T) * norm.cdf(d2)
    elif CallorPut == "put":
        option_value = strike_price * np.exp(-rfr * T) * norm.cdf(-d2) - spot_price * norm.cdf(-d1)
    else:
        raise ValueError(f"Unsupported CallorPut input: {CallorPut}. Use 'call' or 'put'.")
       
    
    return option_value

def PutCallParity(option_price: float, spot_price: float, strike_price: float, T: float, rfr: float, CallorPut: str) -> float:
    """
    Calculate price of put option given a call option and vice-versa.

    Parameters:
        option_price: price of the known option (call or put)
        spot_price: current price of underlying
        strike_price: strike price
        T: time to maturity (in years)
        rfr: risk-free rate
        CallorPut: "call" or "put" (type of the returning option price)

    Returns:
        Corresponding option price (put if input is put, and vice versa)
    """
    CallorPut = CallorPut.lower()
    
    if CallorPut == "put":
        return option_price  + strike_price * math.exp(-rfr * T) - spot_price # + (strike_price / (1 + rfr)**T) - spot_price   # or  * math.exp(-rfr * T)
        
    elif CallorPut == "call":
        return option_price + spot_price -strike_price * math.exp(-rfr * T) #(strike_price / (1 + rfr) ** T)

    else:
        raise ValueError(f"Unsupported CallorPut input: {CallorPut}. Use 'call' or 'put'.")

Before we move on to build our *trading algorithms*, we need to understand when to buy or sell options -- They key lies on the **volatility**. 
Indeed, notice that the only input in our B-S model allowing potential diagreement between traders is exactly volatilikty $\sigma$. As a result, the only way to find a profitable trading strategy is to predict a *future fare volatility*. 

So, one first has to calculate a fair volatility, i.e. the fair price for a call and put options, and then decide by a simple strategy whether to buy or sell that option. In our case, we consider two volatilities:

- The historical volatility (which we described before).
- The implied volatility. 

Let us focus on the latter. In general, the implied volatility, is the volatility one would need to input in the B-S model in order to output its current price. So, if we have the current price (and the rest of needed arguments, e.g. time of expiration, strike price etc.), we just need to solve for $\sigma$. This can be done with many different numerical methods, we choose the standard N-R method (see the ensuing code snipet).

In [108]:
def ImpliedVolatility(option_price: float, spot: float, strike: float, T: float, vol_guess: float, rfr: float, CallorPut: str, approx: bool = False):
    """
    Calculate annualized implied volatility from OHLCV data.

    Parameters: 

    T: time to expiration.
    rfr : Risk-free interest rate (annualized).
    vol_guess: An initial volality --- Usually fetched from Historical Vol.
    Approx.: A boolean --- Since we consider ATM options, we can use a crude approximation for finding IV from call and put options
    Returns:

    A float number-- The implied volatlity
    """

    CallorPut = CallorPut.lower()

    if approx == True:
        if CallorPut == "call":
            sigma = option_price*np.sqrt(2*np.pi)/(spot*np.sqrt(T)) #can also be used as an initializer for N.R.
            return sigma
        elif CallorPut == "put":
            call_price = PutCallParity(option_price, spot, strike, T, rfr, "call")
            if call_price < 0: # means put price << --> violated the no-arbitrage bound
                return "Error"#return "Error"#raise ValueError("Error : Call price is negative")
            else:
                pass
            sigma = call_price*np.sqrt(2*np.pi)/(spot*np.sqrt(T))
            return sigma
    else:
        pass
        

    if CallorPut == "call":
        def price_error(sigma):
            return BlackScholes(spot, strike, T, sigma, rfr, "call") - option_price
    elif CallorPut == "put":
        call_price = PutCallParity(option_price, spot, strike, T, rfr, "call") #Change the put_option to a call_option price
        if call_price < 0:
            return  "Error" 
        else:
            pass
        def price_error(vol_guess):
            return BlackScholes(spot, strike, T, vol_guess, rfr, "call") - call_price
    else:
        raise ValueError(f"Unsupported CallorPut input: {CallorPut}. Use 'call' or 'put'.")

    try:
        implied_volatility = newton(price_error, vol_guess, tol = 1.48e-12)
        return implied_volatility
    except Exception as e:
        return  "Error"  #return "Error" #raise RuntimeError(f"Implied vol convergence failed --- Try approx == True: {e}")

Of course, when trying to build a trading algorithm using the implied volatility (IV) we cannot just get the each quote, find the IV and then trade, since all this does is essentially copy the compentition's inputs, so our market would be very similar, and therfore no trades would be made.

Instead, we update our implied volatility with each new quote, using the cumulative average formula, making a market around this generalised volatility and trade against anyone who disagrees. Specifically, since we are only using ATM quotes, we update our IV w.r.t. the expiration day only (1-D volatility curve), since our strike prices should be roughly the same.

(EXPAND ON DIFFERENT METHODS --- Exponential moving average
                             --- Different initialization instead of HistVol e.g. Garch etc.
)



Once we have calculated our fair volatility, we need to make our market. That is, form a bid and ask around this volatility. This is done by making a *spread* around our fair volality, and using the B-S model and the Put-Call parity principle to calculate our bid (my_bid_price) and ask price (my_ask_price). Once we have our fair prices, we compare them to the market prices. So if my_bid_price > market_ask_price we buy -- completing a trade, and if my_ask_price < market_bid_price we sell  -- completing a trade. That is our strategy for all throughout. 


In the next cell we build the spread and also provide a function which will eventually calculate our profits (or loss) from the buy and sell lists, that we will create using the afore-mentioned strategy. The profit is calculated comparing the closing price of BTC as the option's expirations date.

In [8]:
def BidAsk(spot_price: float, strike_price: float, T: float, volatility: float, rfr: float, CallorPut: str, spread: float) -> list:
    """
    Build a bid–ask spread for options prices using volatility adjustments coming from a fair volatility.
    Buy and sells must still be competitive in today's market. Spread cannot be arbitrarily large.

    Parameters: 

    T: time to expiration.
    rfr : Risk-free interest rate (annualized).

    Returns:

    (Bid_price, Ask_price) 
    """
    
    CallorPut = CallorPut.lower()
    BuyVol = volatility - spread/2  # Recall we use ATM quotes. Usually spreads are not symmetric--- We assume symmetry in this case.
    SellVol = volatility + spread/2 

    TheoryCallBuyPrice = BlackScholes(spot_price, strike_price, T, BuyVol, rfr, "call")
    TheoryPutBuyPrice = PutCallParity(TheoryCallBuyPrice, spot_price, strike_price, T, rfr, "put")

    TheoryCallSellPrice = BlackScholes(spot_price, strike_price, T, SellVol, rfr, "call")
    TheoryPutSellPrice = PutCallParity(TheoryCallSellPrice, spot_price, strike_price, T, rfr, "put")
    
    if CallorPut == "call":
        return TheoryCallBuyPrice, TheoryCallSellPrice
    elif CallorPut == "put":
        return TheoryPutBuyPrice, TheoryPutSellPrice
    else: 
        raise ValueError(f"Unsupported CallorPut input: {CallorPut}. Use 'call' or 'put'.")



def ProfitLoss(BuyList, SellList, month):
    profit = 0
    MoneyMakers = []
    MoneyLosers = []
    expiryPrice = GetCryptoData("BTC/USDT", "1d", 31, month)
    
    for item in BuyList:
        itemDetails = item
        expiryDayPrice = expiryPrice["Close"].iloc[(item[1])]
        if item[0] == "call":
            if expiryDayPrice > item[2]:
                profChange = ((expiryDayPrice - item[2]) - item[3])
                itemDetails.append(profChange)
                if profChange >= 0:
                    #Made money long call
                    MoneyMakers.append(itemDetails)
                    profit += profChange
                else:
                    #lost money long call and exercised (premium too expensive)
                    MoneyLosers.append(itemDetails)
                    profit += profChange
            else:
                profChange = -item[3]
                itemDetails.append(profChange)
                #lost money long call
                MoneyLosers.append(itemDetails)
                profit += profChange
        elif item[0] == "put":
            if expiryDayPrice < item[2]:
                profChange = ((item[2] - expiryDayPrice) - item[3])
                itemDetails.append(profChange)
                if profChange >= 0:
                    #made money long put
                    MoneyMakers.append(itemDetails)
                    profit += profChange
                else:
                    #lost money long call and exercised
                    MoneyLosers.append(itemDetails)
                    profit += profChange
                    
            else:
                profChange = -item[3]
                itemDetails.append(profChange)
                #lost money long put
                MoneyLosers.append(itemDetails)
                profit += profChange
                

    for item in SellList:
        itemDetails = item
        expiryDayPrice = expiryPrice["Close"].iloc[(item[1])]
        if item[0] == "call":
            if expiryDayPrice > item[2]:
                profChange = -((expiryDayPrice - item[2]) - item[3])
                itemDetails.append(profChange)
                if profChange < 0:
                    #lost money short call
                    profit += profChange
                    MoneyLosers.append(itemDetails)
                else:
                    #made money after expiration short call (premium too cheap)
                    profit += profChange
                    MoneyMakers.append(itemDetails)                   
            else:
                #made money short call
                profChange = item[3]
                itemDetails.append(profChange)
                profit += profChange
                MoneyMakers.append(itemDetails)
        elif item[0] == "put":
            if expiryDayPrice < item[2]:
                profChange = -((item[2] - expiryDayPrice) - item[3])
                itemDetails.append(profChange)
                if profChange < 0:
                    #lost money short put
                    profit += profChange
                    MoneyLosers.append(itemDetails)
                else:
                    #made money after having to exercise a short put
                    profit += profChange
                    MoneyMakers.append(itemDetails)
            else:
                profChange = item[3]
                itemDetails.append(profChange)
                #made money short put
                profit += profChange
                MoneyMakers.append(itemDetails)    
    

    return profit, MoneyMakers, MoneyLosers

### Main Algo Trade

Now we present our main algorithms. One using just the historical volatility and the other the implied volality (as described before).

In [107]:
def HistoricalVolTrading(inputYear, inputMonth, spread, longMax, shortMax, rfr):
    current = datetime.strptime(f"{inputYear}-{inputMonth}-01", "%Y-%m-%d")
    yearStamp = current.strftime("%Y-%m-%d") #Input date
    
    prev_day = current - timedelta(days = 1) #Previous month
    prevMonth = datetime(prev_day.year, prev_day.month, 1)
    prevMonthStamp = prevMonth.strftime("%Y-%m-%d")

    file_path = f"D:\Datasets_options_formatted\Format_Dataset_{inputYear}-{inputMonth}_OPTIONS.csv"
    # print(file_path)
    optionData = pd.read_csv(file_path)
    options = []
    BuyList = []
    SellList = []
    currentCallPosition = 0
    currentPutPosition = 0
    i = 0
    dayVolData = GetCryptoData("BTC/USDT", "1d", 30 , prevMonthStamp)
    volatility = HistoricalVolCalc(dayVolData["Close"], dayVolData["Timestamp"],"1d") #Caclulate historical volatility
    #intraday prices for option pricing
    DayBTCPrice = GetCryptoData("BTC/USDT", "5m", 500, yearStamp) #Limit =500 ~ 2 days of price history -- enough to match the options timestamps --- To make BTC to USD

    #Iter through the quotes (sorted by timestamp)
    for i in range(len(optionData)):

        strike = float(optionData.iloc[i]["strike_price"])
        expiration = (float(optionData.iloc[i]["expiration"][8:10]) - 1)/365 #in years --- Since we only use quotes tha expire in the same month this works.
        iDelta = float(optionData.iloc[i]["delta"])
        
        expirationDay = (int(optionData.iloc[i]["expiration"][8:10]) - 1) #expires (#expirationDay) from the first of the month.
        CallorPut = optionData.iloc[i]["type"]

        interval_Timestamp = TimeMinPeriods(optionData["timestamp"][i],5)
        spot = float(DayBTCPrice.iloc[interval_Timestamp]["Close"])
        marketBidPrice = optionData.iloc[i]["bid_price"] * spot #BTC --> USDT BidPrice || 5 min threshold
        marketAskPrice = optionData.iloc[i]["ask_price"] * spot #BTC --> USDT BidPrice || 5 min threshold

        

        AskIV = ImpliedVolatility(marketAskPrice, spot, strike, expiration, volatility, rfr, CallorPut)
        BidIV = ImpliedVolatility(marketBidPrice, spot, strike, expiration, volatility, rfr, CallorPut)
        if AskIV == "Error" or BidIV == "Error":
            continue
            

        if expiration < 0.0028 or math.isnan(marketBidPrice) or math.isnan(marketAskPrice): #1/365 ~ 0.00274 ~ 1 day expiration
            continue #skip invalid or same day expirations
            
        #Create the spread.
        MyBidPrice, MyAskPrice = BidAsk(spot, strike, expiration, volatility, rfr, CallorPut, spread)
        
        
        if CallorPut == "call":
            if marketBidPrice > MyAskPrice and currentCallPosition > -shortMax:
                order = [CallorPut, expirationDay, strike, marketBidPrice, iDelta, 100*BidIV, volatility, "Sell"]
                #create sell order
                SellList.append(order)
                currentCallPosition -= 1
            elif marketAskPrice < MyBidPrice and currentCallPosition < longMax:
                #create buy order
                order = [CallorPut, expirationDay, strike, marketAskPrice, iDelta, 100*AskIV, volatility, "Buy"]
                BuyList.append(order)
                currentCallPosition += 1
        elif CallorPut == "put":
            if marketBidPrice > MyAskPrice and currentPutPosition > -shortMax:
                order = [CallorPut, expirationDay, strike, marketBidPrice, iDelta, 100*BidIV, volatility, "Sell"]
                #create sell order
                SellList.append(order)
                currentPutPosition -= 1
            elif marketAskPrice < MyBidPrice and currentPutPosition < longMax:
                #create buy order
                order = [CallorPut, expirationDay, strike, marketAskPrice, iDelta, 100*AskIV, volatility, "Buy"]
                BuyList.append(order)
                currentPutPosition += 1
        else:
            pass
        
    
    profit, MoneyMakers, MoneyLosers = ProfitLoss(BuyList, SellList, yearStamp)
    print(f"Overall PnL during {inputYear}-{inputMonth} is: ", profit)
    
    return profit, MoneyMakers, MoneyLosers

In [147]:
def ImpliedVolTrading(inputYear, inputMonth, spread, longMax, shortMax, rfr, deltaTrading):
    current = datetime.strptime(f"{inputYear}-{inputMonth}-01", "%Y-%m-%d")
    yearStamp = current.strftime("%Y-%m-%d") #Input date
    
    prev_day = current - timedelta(days = 1) #Previous month
    prevMonth = datetime(prev_day.year, prev_day.month, 1)
    prevMonthStamp = prevMonth.strftime("%Y-%m-%d")
        
    file_path = f"D:\Datasets_options_formatted\Format_Dataset_{inputYear}-{inputMonth}_OPTIONS.csv"
    
    optionData = pd.read_csv(file_path)
    volDataByExpiry = []
    options = []
    BuyList = []
    SellList = []
    currentCallPosition = 0
    currentPutPosition = 0
    #Used for EMA
    alpha = 2/51

    dayVolData = GetCryptoData("BTC/USDT", "1d", 30, prevMonthStamp)
    HistVol = HistoricalVolCalc(dayVolData["Close"], dayVolData["Timestamp"],"1d")
    DayPrice = GetCryptoData("BTC/USDT", "5m", 500, yearStamp)
    for i in range(len(optionData)):

        strike = float(optionData.iloc[i]["strike_price"])
        expiration = (float(optionData.iloc[i]["expiration"][8:10]) - 1)/365  #in years --- Since we only use quotes tha expire in the same month this works.
        iDelta = float(optionData.iloc[i]["delta"])
        expirationDay = (int(optionData.iloc[i]["expiration"][8:10]) - 1) #expires (#expirationDay) from the first of the month.
        CallorPut = optionData.iloc[i]["type"]

        interval_Timestamp = TimeMinPeriods(optionData["timestamp"][i],5)
        spot = float(DayPrice.iloc[interval_Timestamp]["Close"])
        
        marketBidPrice = optionData.iloc[i]["bid_price"] * spot #BTC --> USDT BidPrice || 5 min threshold
        marketAskPrice = optionData.iloc[i]["ask_price"] * spot #BTC --> USDT BidPrice || 5 min threshold

        BidIV = float(optionData.iloc[i]["bid_iv"]) 
        AskIV = float(optionData.iloc[i]["ask_iv"]) 
        MarketIV = (BidIV/2+AskIV/2)
        
        
        
        if expiration < 0.0028 or math.isnan(marketBidPrice) or math.isnan(marketAskPrice):
            continue #to skip same day expirations
        

        AskimpliedVol = ImpliedVolatility(marketAskPrice, spot, strike, expiration, HistVol, rfr, CallorPut)
        BidimpliedVol = ImpliedVolatility(marketBidPrice, spot, strike, expiration, HistVol, rfr, CallorPut)
            
        if AskimpliedVol == "Error" or BidimpliedVol == "Error":
            continue
        MarketImpliedVol = (AskimpliedVol + BidimpliedVol)/2
        
        #adjusting implied volatility
        found = False
        for entry in volDataByExpiry: 
            expiry, avg_vol, count = entry
            if expiry == expirationDay:   #1D volality surface --- i.e. volatility curve --- Recall we trade only ATM options

                # +++++++++++++++ Cumulative average +++++++++++++++
                count += 1
                volatility = ((count-1)/count) * avg_vol + (1/count) *MarketImpliedVol #Try exp. average.
                entry[2] = count
                # +++++++++++++++ Cumulative average +++++++++++++++

                # # +++++++++++++++ Using EMA +++++++++++++++
                # # Compute EMA for historical volatility
                # avg_vol_ema = alpha * MarketImpliedVol + (1 - alpha) * avg_vol
                
                # # Compute EMA for implied volatility
                # volatility = alpha * MarketImpliedVol + (1 - alpha) * avg_vol_ema
                # # +++++++++++++++ Using EMA +++++++++++++++ 

                entry[1] = volatility
                found = True
                break
        if not found:
            volatility = 0.8*HistVol + 0.2*MarketImpliedVol #HistVol*MarketImpliedVol #0.8*HistVol + 0.2*MarketImpliedVol
            volDataByExpiry.append([expirationDay, volatility, 1])
                
        MyBidPrice, MyAskPrice = BidAsk(spot, strike, expiration, volatility, rfr, CallorPut, spread)

        if CallorPut == "call":
            if marketBidPrice > MyAskPrice and currentCallPosition > -shortMax:
                order = [CallorPut, expirationDay, strike, marketBidPrice, iDelta, 100*BidimpliedVol, volatility, "Sell"]
                #create sell order
                SellList.append(order)
                currentCallPosition -= 1
            elif marketAskPrice < MyBidPrice and currentCallPosition < longMax:
                #create buy order
                order = [CallorPut, expirationDay, strike, marketAskPrice, iDelta, 100*AskimpliedVol, volatility, "Buy"]
                BuyList.append(order)
                currentCallPosition += 1
        elif CallorPut == "put":
            if marketBidPrice > MyAskPrice and currentPutPosition > -shortMax:
                order = [CallorPut, expirationDay, strike, marketBidPrice, iDelta, 100*BidimpliedVol, volatility, "Sell"]
                #create sell order
                SellList.append(order)
                currentPutPosition -= 1
            elif marketAskPrice < MyBidPrice and currentPutPosition < longMax:
                #create buy order
                order = [CallorPut, expirationDay, strike, marketAskPrice, iDelta, 100*AskimpliedVol, volatility, "Buy"]
                BuyList.append(order)
                currentPutPosition += 1
        else:
            pass

    if deltaTrading == True:
        deltaProfitBuy = MakeDeltaNeutral(BuyList, inputYear, inputMonth, rfr, False)
        deltaProfitSell = MakeDeltaNeutral(SellList, inputYear, inputMonth, rfr, True)
        deltaProfit = deltaProfitBuy + deltaProfitSell
    else:
        deltaProfit = 0
        
        
    optionProfit, MoneyMakers, MoneyLosers = ProfitLoss(BuyList, SellList, yearStamp)
    profit = optionProfit + deltaProfit
    
    print(f"Overall PnL during {inputYear}-{inputMonth} is: ", profit)
    
    return profit, MoneyMakers, MoneyLosers, volDataByExpiry

### Delta Hedging

Something that we did not touch on is the fact that in our ImpliedVolTrading() function, we also include a boolean which corresponds whether or not we use Delta hedging. 

The following code allows us to provide a daily dynamic Delta hedging (Delta Neutral Trading) by buying or selling the underlying asset (BTC). Each day until the expiration day of the contract option we review the BTC price and buy/sell to revert to a Delta neutral position.

In [11]:
def GroupByExpiration(inputList):
    """
    Groups a list of entries by their expiration day.

    Each entry in the input list is expected to be an iterable (e.g., tuple or list)
    where the expiration day is located at index 1.

    Parameters:
        inputList (list): A list of entries.

    Returns:
        dict: A dictionary where:
              - Keys are expiration days
              - Values are lists of entries that share the same expiration day
    """
    expiration_groups = {}
    for entry in inputList:
        expiration_day = entry[1]  
        if expiration_day not in expiration_groups:
            expiration_groups[expiration_day] = []  
        expiration_groups[expiration_day].append(entry)  
    return expiration_groups

def DeltaCalc(CallorPut, spot_price, strike_price, T, rfr, volatility):
    """ 
    Calculate the \Delta.

    Parameters:
        CallorPut:  Option type. Must be "call" or "put".
        spot_price : Current price of the underlying asset.
        strike_price : Strike price of the option.
        T : Time to maturity --(in days)--.
        volatility : Volatility of the underlying asset (annualized standard deviation).
        rfr : Risk-free interest rate (annualized).
        
    Returns:
        float: The theoretical \Delta.

    Notes
    -----
    Assumes no dividends and constant volatility and interest rates.
    """
    
    T_year = T/365 # Yearly

    d1 = (math.log(spot_price / strike_price) + (rfr + 0.5 * volatility ** 2) * T_year) / (volatility * math.sqrt(T_year))
    
    if CallorPut == 'call':
        delta = math.exp(-rfr * T_year) * norm.cdf(d1)
    elif CallorPut == 'put':
        delta = -math.exp(-rfr * T_year) * norm.cdf(-d1)
    else:
        raise ValueError(f"Unsupported CallorPut input: {CallorPut}. Use 'call' or 'put'.")
    
    return delta



def MakeDeltaNeutral(BuyorSellList: list, year: str, month: str, rfr: float, isSell: bool):
    """
    Simulates daily delta hedging for a set of option positions and computes total delta profit.

    Parameters:
        BuyorSellList (list): A list of option entries.
        
        year (str): Year used to construct the start date for price data.
        month (str): Month used to construct the start date for price data.
        rfr (float): Risk-free rate used in delta calculations.
        isSell (bool): Indicates whether the position is a sell (short) or buy (long). If True -- short.
    """
    total_days = 32
    deltaProfit = 0
    dailyPrices = GetCryptoData("BTC/USDT", "1d", total_days, f"{year}-{month}-01")
    expList = GroupByExpiration(BuyorSellList)
    for key in expList:
        for entry in expList[key]:
            entry.append(deltaProfit) #Appending deltaProfit on each list
               
    
    for today in range(total_days):
        price_today =  dailyPrices["Open"][today]
        for exp_key, entries in sorted(expList.items(), key=lambda item: int(item[0])): 
            if today > exp_key:
                continue
            for entry in entries:
                if today == exp_key:
                    #neutralise your delta positions
                    change_delta = -entry[4] #need to cancel out the delta
                    Money_change = change_delta*price_today 
                    entry[8] += Money_change
                    #We sell our position as option has expired, so we don't need to neutralise it.
                    deltaProfit += entry[8]
                else:
                    if today == 0:
                        delta = DeltaCalc(entry[0], price_today, entry[2], entry[1], rfr, entry[6])
                        if isSell== True:
                            delta = -delta                        
                        entry[4] = delta
                        entry[8] = delta*price_today
                    else: #Hedge dynamical once per day
                        old_delta = entry[4]
                        new_delta = DeltaCalc(entry[0], price_today, entry[2], entry[1], rfr, entry[6])
                        if isSell == True:
                            new_delta = -new_delta
                        change_delta = new_delta - old_delta
                        Money_change = change_delta * price_today
                        entry[8] += Money_change
                        entry[4] = new_delta    
    return deltaProfit

### Here we run our main code:

One can choose which years and months wants to trade -- As long as the appropriate files are download as per DataFormat.ipynb 

Results are shown after we run our code in a tabulate form

In [90]:
def __main__(yearList: list, monthList: list, TradeType: str, spread: float, longPositionMax: float, shortPositionMax: float, rfr: list, deltaTrading: bool):
    year_len = len(yearList)
    profit_df = pd.DataFrame(np.nan,index=months, columns=yearList)
    rfr_index = 0
    for year in yearList:       
        for month in monthList:
            if TradeType == "Historical":
                profit, MoneyMakers, MoneyLosers = HistoricalVolTrading(year, month, spread, longPositionMax, shortPositionMax, rfr[rfr_index])
            elif TradeType == "Implied":
                profit, MoneyMakers, MoneyLosers, _ = ImpliedVolTrading(year, month, spread, longPositionMax, shortPositionMax, rfr[rfr_index], deltaTrading)
            else:
                raise ValueError(f"Wrong Trade type {TradeType}. Use 'Historical' or 'Implied'.")
            all_data = MoneyMakers + MoneyLosers
            print("Number of trades: ", len(all_data))
            profit_df.loc[month,year] = profit
        print("")
        rfr_index+=1
    profit_df.loc["Total"] =  profit_df.sum()
    return profit_df

In [89]:
inputYears = ["2020","2023","2025"]
inputMonths = ["01","02","03","04","05","06","07","08","09","10","11","12"]
rfr = [0.0242, 0.0512, 0.0411]

In [91]:
IV_df_01 = __main__(inputYears, inputMonths, "Implied", 0.2, 20, 10, rfr, True)  #mult

Overall PnL during 2020-01 is:  467.48732907992735
Number of trades:  4
Overall PnL during 2020-02 is:  -37.099526322791235
Number of trades:  5
Overall PnL during 2020-03 is:  -2839.7546144306884
Number of trades:  10
Overall PnL during 2020-04 is:  -2001.9081051959297
Number of trades:  135
Overall PnL during 2020-05 is:  -4754.688697133114
Number of trades:  62
Overall PnL during 2020-06 is:  7515.424856387126
Number of trades:  48
Overall PnL during 2020-07 is:  429.62546280078254
Number of trades:  16
Overall PnL during 2020-08 is:  -685.0536890012281
Number of trades:  36
Overall PnL during 2020-09 is:  4589.2178204172415
Number of trades:  38
Overall PnL during 2020-10 is:  -805.0962451258456
Number of trades:  60
Overall PnL during 2020-11 is:  -2406.5869761470067
Number of trades:  12
Overall PnL during 2020-12 is:  -2299.7222261900533
Number of trades:  53

Overall PnL during 2023-01 is:  -1598.1865052439962
Number of trades:  7
Overall PnL during 2023-02 is:  1398.6095924159

In [80]:
IV_df_02 = __main__(inputYears, inputMonths, "Implied", 0.2, 20, 10, rfr, True)  #sum -- w_1 = 0.8, w_2 = 0.2

Overall PnL during 2020-01 is:  0
Number of trades:  0
Overall PnL during 2020-02 is:  0
Number of trades:  0
Overall PnL during 2020-03 is:  1712.4938575647002
Number of trades:  7
Overall PnL during 2020-04 is:  -1866.7376063985516
Number of trades:  131
Overall PnL during 2020-05 is:  -3776.698873328721
Number of trades:  54
Overall PnL during 2020-06 is:  9681.984122493814
Number of trades:  58
Overall PnL during 2020-07 is:  307.91750486522085
Number of trades:  8
Overall PnL during 2020-08 is:  1082.402840598471
Number of trades:  36
Overall PnL during 2020-09 is:  4084.2306548520573
Number of trades:  38
Overall PnL during 2020-10 is:  -431.5500763465443
Number of trades:  62
Overall PnL during 2020-11 is:  -921.1724378810409
Number of trades:  4
Overall PnL during 2020-12 is:  -3556.024441085363
Number of trades:  51

Overall PnL during 2023-01 is:  0
Number of trades:  0
Overall PnL during 2023-02 is:  -994.2954607381193
Number of trades:  9
Overall PnL during 2023-03 is:  0
N

In [ ]:
HV_df_01 = __main__(inputYears, inputMonths, "Historical", 0.2, 20, 10, rfr, True) 

In [133]:
IV_df_03 = __main__(inputYears, inputMonths, "Implied", 0.2, 20, 10, rfr, True)  #Hist initial

Overall PnL during 2020-01 is:  0
Number of trades:  0
Overall PnL during 2020-02 is:  0
Number of trades:  0
Overall PnL during 2020-03 is:  440.82789950283336
Number of trades:  6
Overall PnL during 2020-04 is:  -1418.786765102947
Number of trades:  137
Overall PnL during 2020-05 is:  -3548.5904130472804
Number of trades:  54
Overall PnL during 2020-06 is:  9258.0490772633
Number of trades:  56
Overall PnL during 2020-07 is:  237.75147680091294
Number of trades:  7
Overall PnL during 2020-08 is:  1324.1870169282029
Number of trades:  38
Overall PnL during 2020-09 is:  4443.1849841066905
Number of trades:  40
Overall PnL during 2020-10 is:  -256.9807921312022
Number of trades:  64
Overall PnL during 2020-11 is:  -767.4747643257215
Number of trades:  4
Overall PnL during 2020-12 is:  -4658.772886963401
Number of trades:  51

Overall PnL during 2023-01 is:  82.96349906846154
Number of trades:  1
Overall PnL during 2023-02 is:  688.7673877144969
Number of trades:  12
Overall PnL during 2

In [135]:
IV_df_03_EMA = __main__(inputYears, inputMonths, "Implied", 0.2, 20, 10, rfr, True) 

Overall PnL during 2020-01 is:  0
Number of trades:  0
Overall PnL during 2020-02 is:  0
Number of trades:  0
Overall PnL during 2020-03 is:  -650.8778923694044
Number of trades:  11
Overall PnL during 2020-04 is:  3106.9797175499025
Number of trades:  166
Overall PnL during 2020-05 is:  -11474.513198268354
Number of trades:  77
Overall PnL during 2020-06 is:  12662.838246532523
Number of trades:  58
Overall PnL during 2020-07 is:  628.7668832790722
Number of trades:  10
Overall PnL during 2020-08 is:  -2230.4787987072923
Number of trades:  42
Overall PnL during 2020-09 is:  6773.4970335129165
Number of trades:  81
Overall PnL during 2020-10 is:  2599.283973218651
Number of trades:  63
Overall PnL during 2020-11 is:  -4670.0054739386
Number of trades:  20
Overall PnL during 2020-12 is:  1848.5480223445047
Number of trades:  90

Overall PnL during 2023-01 is:  143.91057720637514
Number of trades:  2
Overall PnL during 2023-02 is:  2675.6312063333316
Number of trades:  15
Overall PnL dur

In [140]:
IV_df_04 = __main__(inputYears, inputMonths, "Implied", 0.2, 20, 10, rfr, True)  #w_1 = 0.1 , w_2 = 0.9

Overall PnL during 2020-01 is:  0
Number of trades:  0
Overall PnL during 2020-02 is:  0
Number of trades:  0
Overall PnL during 2020-03 is:  441.3146261116151
Number of trades:  6
Overall PnL during 2020-04 is:  -1541.9967610484869
Number of trades:  135
Overall PnL during 2020-05 is:  -3549.7024414207635
Number of trades:  54
Overall PnL during 2020-06 is:  9678.929707257135
Number of trades:  58
Overall PnL during 2020-07 is:  307.41696813131637
Number of trades:  8
Overall PnL during 2020-08 is:  1378.415297105481
Number of trades:  38
Overall PnL during 2020-09 is:  4090.7417829624137
Number of trades:  38
Overall PnL during 2020-10 is:  -417.30643955073174
Number of trades:  62
Overall PnL during 2020-11 is:  -846.8858773102056
Number of trades:  4
Overall PnL during 2020-12 is:  -3620.658418044375
Number of trades:  51

Overall PnL during 2023-01 is:  59.30979457181718
Number of trades:  1
Overall PnL during 2023-02 is:  -293.19011820581727
Number of trades:  11
Overall PnL duri

# ------- Results -------

Below we show the results for different methods used. The results include monthly profit/loss for 3 years. 

- We want to note the large difference in --HistVol vs ImpliedVol-- profits during 2025, which seem to be much lower for the ImpliedVol algo. We believe this suggest that  during this year, the market inaccurately accounted for market volatility.
- Building on the previous observation, by reducing the influence of the market vol. seemed to produce more profit for that year (2025) during our experiments.

In [95]:
#We used the Historical volatility algo.
print(tabulate(HV_df_01, headers="keys", tablefmt="grid"))

+-------+-----------+------------+-----------+
|       |      2020 |       2023 |      2025 |
+=======+===========+============+===========+
| 01    |      0    |   6065.51  |  25360.2  |
+-------+-----------+------------+-----------+
| 02    |  -7406.79 |   4511.59  | -13021.3  |
+-------+-----------+------------+-----------+
| 03    |   8145.87 |   -334.136 |   2027.07 |
+-------+-----------+------------+-----------+
| 04    |  -1156    | -27900.9   | -73573.1  |
+-------+-----------+------------+-----------+
| 05    |   5594.13 |   -931.189 |  47919.1  |
+-------+-----------+------------+-----------+
| 06    |   2968.71 |   6171.48  |  44914.2  |
+-------+-----------+------------+-----------+
| 07    | -15908.4  |  -1583.51  | -32868.7  |
+-------+-----------+------------+-----------+
| 08    |   4913.26 |   9243.63  |  29766.1  |
+-------+-----------+------------+-----------+
| 09    |  38222.8  | -13006.4   |   2068.92 |
+-------+-----------+------------+-----------+
| 10    |  -9

In [97]:
#We used the Implied volatility algo. with a cum. avarage smoothing (initial vol = MarketIV * HistIV) with Dynamical Delta Hedging
print(tabulate(IV_df_01, headers="keys", tablefmt="grid")) #mult

+-------+------------+----------+----------+
|       |       2020 |     2023 |     2025 |
+=======+============+==========+==========+
| 01    |   467.487  | -1598.19 |  8649.6  |
+-------+------------+----------+----------+
| 02    |   -37.0995 |  1398.61 | -2517.83 |
+-------+------------+----------+----------+
| 03    | -2839.75   | -3329.1  | -3078.87 |
+-------+------------+----------+----------+
| 04    | -2001.91   |  7205.7  |  3648.87 |
+-------+------------+----------+----------+
| 05    | -4754.69   |  3675.78 |  4858.98 |
+-------+------------+----------+----------+
| 06    |  7515.42   | -3120.76 | 16976.8  |
+-------+------------+----------+----------+
| 07    |   429.625  |  4016.07 | -3171.39 |
+-------+------------+----------+----------+
| 08    |  -685.054  |  4031.39 |  7255.72 |
+-------+------------+----------+----------+
| 09    |  4589.22   |  1140.88 | 18381.1  |
+-------+------------+----------+----------+
| 10    |  -805.096  | -7031.71 | -9309.65 |
+-------+-

In [96]:
#We used the Implied volatility algo. with a cum. avarage smoothing (initial vol = w_1*MarketIV  + w_2*HistIV) with Dynamical Delta Hedging
                                                                    # w_1 = 0.2 , w_2 = 0.8
print(tabulate(IV_df_02, headers="keys", tablefmt="grid"))

+-------+-----------+-----------+-----------+
|       |      2020 |      2023 |      2025 |
+=======+===========+===========+===========+
| 01    |     0     |     0     |  7156.69  |
+-------+-----------+-----------+-----------+
| 02    |     0     |  -994.295 |     0     |
+-------+-----------+-----------+-----------+
| 03    |  1712.49  |     0     |     0     |
+-------+-----------+-----------+-----------+
| 04    | -1866.74  |  4979.42  | 12108.1   |
+-------+-----------+-----------+-----------+
| 05    | -3776.7   |  5680.21  |   542.676 |
+-------+-----------+-----------+-----------+
| 06    |  9681.98  |  4377.33  |  -874.116 |
+-------+-----------+-----------+-----------+
| 07    |   307.918 |     0     | -1311.49  |
+-------+-----------+-----------+-----------+
| 08    |  1082.4   |  7209.86  |  2994.9   |
+-------+-----------+-----------+-----------+
| 09    |  4084.23  |   787.127 |  4762.14  |
+-------+-----------+-----------+-----------+
| 10    |  -431.55  | -5399.87  | 

In [141]:
#We used the Implied volatility algo. with a cum. avarage smoothing (initial vol = w_1*MarketIV  + w_2*HistIV) with Dynamical Delta Hedging
                                                                    # w_1 = 0.1 , w_2 = 0.9
print(tabulate(IV_df_04, headers="keys", tablefmt="grid"))

+-------+-----------+------------+-----------+
|       |      2020 |       2023 |      2025 |
+=======+===========+============+===========+
| 01    |     0     |    59.3098 |  7167.74  |
+-------+-----------+------------+-----------+
| 02    |     0     |  -293.19   |     0     |
+-------+-----------+------------+-----------+
| 03    |   441.315 |     0      |     0     |
+-------+-----------+------------+-----------+
| 04    | -1542     |  4965.11   | 12041.3   |
+-------+-----------+------------+-----------+
| 05    | -3549.7   |  5396.59   |  1110.41  |
+-------+-----------+------------+-----------+
| 06    |  9678.93  |  4379.35   |  -874.096 |
+-------+-----------+------------+-----------+
| 07    |   307.417 |     0      | -3750.21  |
+-------+-----------+------------+-----------+
| 08    |  1378.42  |  7211.05   |  2864.14  |
+-------+-----------+------------+-----------+
| 09    |  4090.74  |   787.067  |  4762.53  |
+-------+-----------+------------+-----------+
| 10    |  -4

In [136]:
#We used the Implied volatility algo. with a cum. avarage smoothing (initial vol = Hist) with Dynamical Delta Hedging
print(tabulate(IV_df_03, headers="keys", tablefmt="grid"))

+-------+-----------+------------+-----------+
|       |      2020 |       2023 |      2025 |
+=======+===========+============+===========+
| 01    |     0     |    82.9635 |  7179.45  |
+-------+-----------+------------+-----------+
| 02    |     0     |   688.767  |     0     |
+-------+-----------+------------+-----------+
| 03    |   440.828 |     0      |     0     |
+-------+-----------+------------+-----------+
| 04    | -1418.79  |  3616.78   | 11977     |
+-------+-----------+------------+-----------+
| 05    | -3548.59  |  5416.63   | -2568.53  |
+-------+-----------+------------+-----------+
| 06    |  9258.05  |  4381.37   |  -300.569 |
+-------+-----------+------------+-----------+
| 07    |   237.751 |  -279.101  | -2397.64  |
+-------+-----------+------------+-----------+
| 08    |  1324.19  |  7212.24   | -1488.35  |
+-------+-----------+------------+-----------+
| 09    |  4443.18  |   787.007  |  4762.93  |
+-------+-----------+------------+-----------+
| 10    |  -2

In [137]:
#We used the Implied volatility algo. with a Exp Weights Av. (EMA) smoothing (initial vol = Hist) with Dynamical Delta Hedging
print(tabulate(IV_df_03_EMA, headers="keys", tablefmt="grid"))

+-------+------------+------------+-----------+
|       |       2020 |       2023 |      2025 |
+=======+============+============+===========+
| 01    |      0     |    143.911 |  15392.9  |
+-------+------------+------------+-----------+
| 02    |      0     |   2675.63  |      0    |
+-------+------------+------------+-----------+
| 03    |   -650.878 |      0     |      0    |
+-------+------------+------------+-----------+
| 04    |   3106.98  |  -6228.36  |  17379.2  |
+-------+------------+------------+-----------+
| 05    | -11474.5   |  -7390.75  |   6785.35 |
+-------+------------+------------+-----------+
| 06    |  12662.8   |   6602.71  |   2541.52 |
+-------+------------+------------+-----------+
| 07    |    628.767 |   -324.505 |  -4819.13 |
+-------+------------+------------+-----------+
| 08    |  -2230.48  |   3886.33  |   3061.59 |
+-------+------------+------------+-----------+
| 09    |   6773.5   |    213.131 |   5992.72 |
+-------+------------+------------+-----

# ====== Some testing ======= (Not final)

### For different spreads 

In [99]:
def testy(yearList: list, monthList: list, TradeType: str, spread: float, longPositionMax: float, shortPositionMax: float, deltaTrading: bool):
    year_len = len(yearList)
    rfr=0.0411 #0.0411#0.0512
    months = ["01","02","03","04","05","06","07","08","09","10","11","12"]
    profit_df = pd.DataFrame(np.nan,index=months, columns=yearList)
    # profit_df = profit_df.replace("N/A", pd.NA).astype("float")
    #print(profit_df.loc[:,"2020"]) 
    # print(profit_df.iloc[:,0])
    # print(profit_df)
    for year in yearList:       
        for month in monthList:
            if TradeType == "Historical":
                profit, MoneyMakers, MoneyLosers = HistoricalVolTrading(year, month, spread, longPositionMax, shortPositionMax, rfr)
            elif TradeType == "Implied":
                profit, MoneyMakers, MoneyLosers, _ = ImpliedVolTrading(year, month, spread, longPositionMax, shortPositionMax, rfr, deltaTrading)
            else:
                raise ValueError(f"Wrong Trade type {TradeType}. Use 'Historical' or 'Implied'.")
            profit_df.loc[month,year] = profit
        print("")
    profit_df.loc["Total"] =  profit_df.sum()
    return profit_df

In [148]:
monthsss = ["01","02","03","04","05","06","07","08","09","10","11","12"]
__main__(["2025"], monthsss, "Implied", 0.1, 20, 10, [0.0411], True)

Overall PnL during 2025-01 is:  15493.791978550329
Number of trades:  11
Overall PnL during 2025-02 is:  -385.24897509420316
Number of trades:  1
Overall PnL during 2025-03 is:  -39181.50213435669
Number of trades:  10
Overall PnL during 2025-04 is:  -4623.64840931943
Number of trades:  38
Overall PnL during 2025-05 is:  20009.713452659467
Number of trades:  30
Overall PnL during 2025-06 is:  7447.140662557929
Number of trades:  9
Overall PnL during 2025-07 is:  -1773.1895667488352
Number of trades:  6
Overall PnL during 2025-08 is:  -8585.308362370883
Number of trades:  39
Overall PnL during 2025-09 is:  27761.9399278865
Number of trades:  65
Overall PnL during 2025-10 is:  21928.50325289818
Number of trades:  21
Overall PnL during 2025-11 is:  -116.41576825270567
Number of trades:  3
Overall PnL during 2025-12 is:  24754.79944355623
Number of trades:  19



,2025
01,15493.791979
02,-385.248975
03,-39181.502134
04,-4623.648409
05,20009.713453
06,7447.140663
07,-1773.189567
08,-8585.308362
09,27761.939928
10,21928.503253


In [149]:
monthsss = ["01","02","03","04","05","06","07","08","09","10","11","12"]
__main__(["2025"], monthsss, "Implied", 0.15, 20, 10, [0.0411], True)

Overall PnL during 2025-01 is:  9176.527085455233
Number of trades:  6
Overall PnL during 2025-02 is:  -385.24897509420316
Number of trades:  1
Overall PnL during 2025-03 is:  -23258.783669302953
Number of trades:  4
Overall PnL during 2025-04 is:  3311.6952439672823
Number of trades:  32
Overall PnL during 2025-05 is:  -721.0525008754785
Number of trades:  10
Overall PnL during 2025-06 is:  -229.28769693866934
Number of trades:  2
Overall PnL during 2025-07 is:  -2276.631366227306
Number of trades:  5
Overall PnL during 2025-08 is:  3335.148467393814
Number of trades:  10
Overall PnL during 2025-09 is:  2509.889064893425
Number of trades:  36
Overall PnL during 2025-10 is:  -4398.854556224129
Number of trades:  16
Overall PnL during 2025-11 is:  -1773.4748648092939
Number of trades:  1
Overall PnL during 2025-12 is:  -502.4437268576196
Number of trades:  12



,2025
01,9176.527085
02,-385.248975
03,-23258.783669
04,3311.695244
05,-721.052501
06,-229.287697
07,-2276.631366
08,3335.148467
09,2509.889065
10,-4398.854556


In [150]:
monthsss = ["01","02","03","04","05","06","07","08","09","10","11","12"]
__main__(["2025"], monthsss, "Implied", 0.125, 20, 10, [0.0411], True)

Overall PnL during 2025-01 is:  10849.322918897771
Number of trades:  8
Overall PnL during 2025-02 is:  -385.24897509420316
Number of trades:  1
Overall PnL during 2025-03 is:  -29637.837527026513
Number of trades:  7
Overall PnL during 2025-04 is:  -9797.909506591
Number of trades:  34
Overall PnL during 2025-05 is:  12047.462897599908
Number of trades:  22
Overall PnL during 2025-06 is:  781.1291215945084
Number of trades:  6
Overall PnL during 2025-07 is:  -1773.1895667488352
Number of trades:  6
Overall PnL during 2025-08 is:  5992.862486628363
Number of trades:  23
Overall PnL during 2025-09 is:  10242.745866804613
Number of trades:  48
Overall PnL during 2025-10 is:  12074.679013536119
Number of trades:  19
Overall PnL during 2025-11 is:  -1773.4748648092939
Number of trades:  1
Overall PnL during 2025-12 is:  10385.144249994102
Number of trades:  16



,2025
01,10849.322919
02,-385.248975
03,-29637.837527
04,-9797.909507
05,12047.462898
06,781.129122
07,-1773.189567
08,5992.862487
09,10242.745867
10,12074.679014
